# Human 배치 — 보드 입력 스케일링 비교
## 1. 왜 / 무엇
L2를 강화했을 때 NLL은 낮아졌지만 배치 정확도는 개선되지 않았습니다.
이번에는 **보드 픽셀의 표준편차 나눗셈만 제거**합니다. 평균 중심화는 유지합니다.

| 조건 | 보드200차원 | 조각7차원 |
|---|---|---|
| standardized | (x − Train 평균) / Train 표준편차 | 기존 표준화 |
| board_centered | x − Train 평균 | 기존 표준화 그대로 |

모델·alpha0.0001·30epoch·데이터/정답은 같습니다. 별도 Smoke 학습은 없습니다.
[명세](../../../ai-harness/ml-experiments/experiments/human-placement-board-scaling.md)


## 2. 손으로 이해하기
어떤 칸이 Train에서0.047%만 채워졌다면 평균은 약0.00047, 표준편차는0.0216입니다.
그 칸이 채워졌을 때 기존 값은 (1−0.00047)/0.0216 ≈46.2, 비교 조건은1−0.00047 ≈0.9995입니다.

이는 표준화의 정상 계산입니다. 큰 값이 반드시 실패 원인은 아닙니다.
기존 NLL14.4 게임에는0행 점유가 없었으므로 그 실패를 이 값 탓으로 확정하지 않습니다.
같은 조건에서 실제 정확도/NLL이 개선되는지 검증해야 합니다.


In [ ]:
import contextlib
import hashlib
import io
import json
import os
import platform
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ai-harness").is_dir() and (p / "package.json").is_file())
EXPERIMENT_ID = "human-placement-board-scaling"
SOURCE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_placement_board_scaling.ipynb"
PREVIOUS_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_placement_l2_regularization.ipynb"
PREVIOUS_DEFS_SHA = "9112dedc65f7b4d5f04ef2f0107561914be53da022f7f7e51dc7e50d9ce60cbd"
REFERENCE_PATH = ROOT / "data/ml-experiments/human-placement-l2-regularization/full-report-run1.json"
REFERENCE_SHA = "1b3d28aca7319265e8a73c06629ca2473d6dd7597b2967ccf78be5135491779e"
selected = [{"id": c["id"], "source": "".join(c["source"])}
            for c in json.loads(PREVIOUS_NOTEBOOK.read_text(encoding="utf-8"))["cells"]
            if c["cell_type"] == "code" and "definitions" in c.get("metadata", {}).get("tags", [])]
assert hashlib.sha256((json.dumps(selected, ensure_ascii=False, sort_keys=True,
                                  allow_nan=False, indent=2) + "\n").encode()).hexdigest() == PREVIOUS_DEFS_SHA
previous = {"__name__": "board_scaling_previous"}
for c in selected:
    exec(compile(c["source"], str(PREVIOUS_NOTEBOOK), "exec"), previous)
shared = previous["shared"]
shared["EXPERIMENT_ID"], shared["SOURCE_NOTEBOOK"] = EXPERIMENT_ID, SOURCE_NOTEBOOK
shared["base"]["dependency"]["link"]["dataset"]["shared"]["EXPERIMENT_ID"] = EXPERIMENT_ID
for name in ("require", "json_bytes", "sha256_file", "definition_sha", "threadpool_limits",
             "report_path", "publish_report", "fit_candidate", "close_numeric"):
    globals()[name] = previous[name]
VARIANTS = ("standardized", "board_centered")
ALPHA, EPOCHS = 0.0001, 30


In [ ]:
def transform_features(raw, scaler, variant):
    """같은 Train 통계를 사용하고 candidate의 보드 분산 나눗셈만 제거한다."""
    require(variant in VARIANTS and raw.ndim == 2 and raw.shape[1] == 207
            and np.isfinite(raw).all(), "INVALID_FEATURES_OR_VARIANT")
    result = scaler.transform(raw)
    if variant == "board_centered":
        result[:, :200] = raw[:, :200] - scaler.mean_[:200]
    require(np.isfinite(result).all(), "NONFINITE_TRANSFORM")
    return result

def effective_scale(scaler, variant):
    require(variant in VARIANTS, "INVALID_VARIANT")
    scale = scaler.scale_.copy()
    if variant == "board_centered":
        scale[:200] = 1
    return scale.tolist()

def input_summary(values):
    board = np.abs(values[:, :200])
    return {"maxAbsBoard": float(board.max()), "p99AbsBoard": float(np.quantile(board, .99)),
            "maxAbsPiece": float(np.abs(values[:, 200:]).max())}

def load_reference():
    require(sha256_file(REFERENCE_PATH) == REFERENCE_SHA, "REFERENCE_SHA_MISMATCH")
    ref = json.loads(REFERENCE_PATH.read_text(encoding="utf-8"), parse_constant=shared["reject_constant"])
    require(ref["experimentId"] == "human-placement-l2-regularization"
            and ref["runMode"] == "full" and ref["reportSchemaVersion"] == 1
            and ref["fullCohortCovered"] is True and ref["baselineReproduced"] is True
            and ref["sourceDefinitionSha256"] == PREVIOUS_DEFS_SHA
            and ref["snapshotDefinitionsSha256"] == previous["SNAPSHOT_DEFS_SHA"], "REFERENCE_CONTRACT_MISMATCH")
    return ref


In [ ]:
def run_full():
    require(VARIANTS == ("standardized", "board_centered") and ALPHA == .0001 and EPOCHS == 30,
            "EXPERIMENT_SETTINGS_CHANGED")
    require((platform.python_version(), np.__version__, shared["base"]["sklearn"].__version__)
            == ("3.12.13", "2.5.1", "1.9.0"), "PINNED_ENVIRONMENT_REQUIRED")
    source_sha = definition_sha(SOURCE_NOTEBOOK)
    reference = load_reference()
    data, watched = shared["load_snapshots"]()
    vocabulary = shared["train_vocabulary"](data["train"])
    require(len(vocabulary) == 2133, "TRAIN_VOCABULARY_CHANGED")
    ids = {key: i for i, key in enumerate(vocabulary)}
    labels = np.array([ids[r["targetKey"]] for r in data["train"]], dtype=int)
    raw = {s: np.stack([r["x"] for r in data[s]]) for s in ("train", "validation")}
    scaler = shared["base"]["StandardScaler"]().fit(raw["train"])
    require(np.allclose(scaler.mean_, reference["training"]["scalerMean"], rtol=0, atol=1e-12)
            and np.allclose(scaler.scale_, reference["training"]["scalerScale"], rtol=0, atol=1e-12)
            and hashlib.sha256(json_bytes(vocabulary)).hexdigest() == reference["vocabulary"]["sha256"],
            "PREPROCESSING_DRIFT")
    results, diagnostics = {}, {}
    with threadpool_limits(limits=1):
        for variant in VARIANTS:
            transformed = {s: transform_features(raw[s], scaler, variant) for s in raw}
            diagnostics[variant] = {s: input_summary(x) for s, x in transformed.items()}
            print(variant)
            results[variant] = fit_candidate(transformed["train"], labels, transformed["validation"],
                data["train"], data["validation"], vocabulary, ALPHA, EPOCHS)
            require(all(h["train"]["knownNllDenominator"] == 6417
                        and h["validation"]["segments"] == 1516
                        and h["validation"]["knownNllDenominator"] == 1364
                        and h["validation"]["unknownTargetSegments"] == 152
                        for h in results[variant]["history"]), "METRIC_DENOMINATOR_CHANGED")
            if variant == "standardized":
                require(close_numeric(results[variant], reference["results"]["alpha_0.0001"]),
                        "BASELINE_REPRODUCTION_FAILED")
                print("Baseline reproduced within 1e-12. Continuing board-centered comparison.")
    baseline, candidate = (results[v]["result"] for v in VARIANTS)
    delta = {m: candidate["gameMacro"][m] - baseline["gameMacro"][m] for m in ("top1", "top3", "mrr")}
    delta["knownNll"] = candidate["segmentMicro"]["knownTargetNll"] - baseline["segmentMicro"]["knownTargetNll"]
    per_game = []
    for a, b in zip(baseline["perGame"], candidate["perGame"]):
        require(a["gameId"] == b["gameId"], "GAME_ORDER_CHANGED")
        per_game.append({"gameId": a["gameId"], "segments": a["segments"],
                         **{"delta_" + m: b[m] - a[m] for m in ("top1", "top3", "mrr")}})
    require(all(sha256_file(p) == h for p, h in watched.items())
            and sha256_file(REFERENCE_PATH) == REFERENCE_SHA
            and definition_sha(SOURCE_NOTEBOOK) == source_sha
            and definition_sha(PREVIOUS_NOTEBOOK) == PREVIOUS_DEFS_SHA
            and definition_sha(previous["SNAPSHOT_NOTEBOOK"]) == previous["SNAPSHOT_DEFS_SHA"]
            and definition_sha(shared["BASE_NOTEBOOK"]) == shared["BASE_DEFS_SHA"], "INPUT_OR_CODE_CHANGED")
    return {"reportSchemaVersion": 1, "experimentId": EXPERIMENT_ID, "runMode": "full",
        "executionPolicy": "full-only", "fullCohortCovered": True, "policyDecisionEligible": True,
        "canonicalResultEligible": False, "productionIntegration": False, "inferenceEligible": False,
        "wholePolicyEvidence": False, "legalityChecked": False, "reachableCandidatesGenerated": False,
        "snapshotObservation": "unverified", "analysisStatus": "board_scaling_comparison_requires_review",
        "modelFitCount": 2, "smokeFitCount": 0, "variants": list(VARIANTS), "results": results,
        "candidateMinusBaseline": delta, "perGameDelta": per_game, "inputDiagnostics": diagnostics,
        "effectiveScaleByVariant": {v: effective_scale(scaler, v) for v in VARIANTS},
        "baselineReproduced": True, "baselineTolerance": 1e-12, "bestEpochSelected": False,
        "validationRole": "reused_development_comparison_not_independent_test",
        "training": {**reference["training"], "alpha": ALPHA,
                     "scalerMean": scaler.mean_.tolist(), "scalerScale": scaler.scale_.tolist()},
        "vocabulary": reference["vocabulary"], "provenance": reference["provenance"],
        "epochMetricContract": reference["epochMetricContract"], "environment": reference["environment"],
        "sourceDefinitionSha256": source_sha, "previousDefinitionsSha256": PREVIOUS_DEFS_SHA,
        "referenceSha256": REFERENCE_SHA, "sourceUnchanged": True, "testFileRead": False,
        "rawFileRead": False, "newConditionRepeatDeterminismChecked": False}


In [ ]:
def internal_checks():
    """학습 대신 공유 가짜 모델 검사와 작은 배열의 전처리 계약을 확인한다."""
    with contextlib.redirect_stdout(io.StringIO()):
        previous["internal_checks"]()
    raw = np.zeros((4, 207))
    raw[0, 0], raw[1, 1] = 1, 1
    raw[:, 200] = [0, 0, 1, 1]
    frozen = raw.copy()
    scaler = shared["base"]["StandardScaler"]().fit(raw)
    mean, scale = scaler.mean_.copy(), scaler.scale_.copy()
    standard = transform_features(raw, scaler, "standardized")
    centered = transform_features(raw, scaler, "board_centered")
    assert np.array_equal(standard, scaler.transform(raw))
    assert np.array_equal(centered[:, :200], raw[:, :200] - mean[:200])
    assert np.array_equal(centered[:, 200:], standard[:, 200:])
    assert np.max(np.abs(centered[:, :200])) <= 1 and standard[0, 0] > centered[0, 0]
    assert np.array_equal(raw, frozen) and np.array_equal(scaler.mean_, mean) and np.array_equal(scaler.scale_, scale)
    assert np.array_equal(np.array(effective_scale(scaler, "board_centered"))[:200], np.ones(200))
    assert np.array_equal(np.array(effective_scale(scaler, "board_centered"))[200:], scale[200:])
    assert np.array_equal(np.array(effective_scale(scaler, "standardized")), scale)
    assert not centered[:, 2].any() and scaler.scale_[2] == 1
    validation = np.ones((1, 207))
    result = transform_features(validation, scaler, "board_centered")
    assert np.array_equal(result[:, :200], validation[:, :200] - mean[:200])
    assert np.array_equal(scaler.mean_, mean) and np.array_equal(scaler.scale_, scale)
    bad = raw.copy(); bad[0, 0] = np.nan
    for matrix, variant in ((bad, "standardized"), (raw[:, :206], "standardized"), (raw, "unknown")):
        try:
            transform_features(matrix, scaler, variant)
        except ValueError:
            pass
        else:
            raise AssertionError("INVALID_TRANSFORM_INPUT_ACCEPTED")
    return "PASS: shared training/ranking/path contracts; board-only variance removal; piece features identical; Train stats immutable; zero variance; finite/shape/variant checks. No real fit."


In [ ]:
def show_report(report):
    require(report["experimentId"] == EXPERIMENT_ID and report["baselineReproduced"], "REPORT_CONTRACT_INVALID")
    summary, diagnostic = [], []
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    for col, variant in enumerate(VARIANTS):
        item = report["results"][variant]; history = item["history"]
        summary.append({"variant": variant,
            **{"valMacro_" + m: item["result"]["gameMacro"][m] for m in ("top1", "top3", "mrr")},
            "valKnownNll": item["result"]["segmentMicro"]["knownTargetNll"]})
        diagnostic.append({"variant": variant, "trainTop1": history[-1]["train"]["top1"],
            "valTop1": history[-1]["validation"]["top1"], "trainNll": history[-1]["train"]["knownTargetNll"],
            "valKnownNll": history[-1]["validation"]["knownTargetNll"], "coefficientL2": item["coefficientL2"]})
        for split, style in (("train", "-"), ("validation", "--")):
            last = history[-1][split]; x = [h["epoch"] for h in history]
            axes[0, col].plot(x, [h[split]["knownTargetNll"] for h in history], style,
                             label=f"{split} known n={last['knownNllDenominator']}/{last['segments']}")
            axes[1, col].plot(x, [h[split]["top1"] for h in history], style, label=split)
        axes[0, col].set(title=variant, xlabel="epoch", ylabel="Known-target NLL, no L2")
        axes[1, col].set(xlabel="epoch", ylabel="Top-1, all targets", ylim=(0, 1))
        axes[0, col].legend(fontsize=8); axes[1, col].legend()
    print(report["analysisStatus"], "| development comparison, NOT gameplay evidence")
    display(pd.DataFrame(summary).set_index("variant"))
    display(pd.DataFrame([report["candidateMinusBaseline"]], index=["board_centered - standardized"]))
    display(pd.DataFrame(diagnostic).set_index("variant"))
    display(pd.DataFrame([{"variant": v, "split": s, **values}
            for v, splits in report["inputDiagnostics"].items() for s, values in splits.items()]))
    print("Val NLL: known1364; accuracy: all1516 including152 OOV. Input/L2 effects interact.")
    fig.tight_layout(); plt.show(); plt.close(fig)
    games = pd.DataFrame(report["perGameDelta"])
    display(games)
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(range(len(games)), games["delta_top1"])
    ax.axhline(0, color="black", linewidth=.8)
    ax.set(xlabel="Game index (Unicode order)", ylabel="Top-1 delta; positive is higher",
           title="board_centered minus standardized")
    fig.tight_layout(); plt.show(); plt.close(fig)


## 3. 내부 검사 / Full 실행
기본 모두 실행은 실제 모델 학습 없이 내부 검사만 수행합니다.
Full을 실행하려면 아래 RUN_FULL=True로 바꾸고 새 FULL_REPORT 경로를 지정하세요.
두 조건×30epoch이며, 기준선 재현 실패 시 candidate를 학습하지 않습니다.
기존 Notebook/보고서는 덮어쓰지 않습니다. 새 Notebook을 열고 위에서부터 실행하면 됩니다.


In [ ]:
RUN_FULL = False
FULL_REPORT = os.environ.get("HUMAN_PLACEMENT_BOARD_SCALING_REPORT",
    "data/ml-experiments/human-placement-board-scaling/full-report-run1.json")


In [ ]:
print(internal_checks())
full_report = None
if RUN_FULL:
    destination = report_path(FULL_REPORT)
    full_report = run_full()
    publish_report(full_report, destination)
    print("Saved:", str(destination), "| SHA-256:", sha256_file(destination))
else:
    print("Full disabled. No real dataset/reference report access or neural training.")


## 4. 결과 읽기
실행 후 아래 표/그림을 확인합니다. 결과만 다시 볼 때는 RUN_FULL=False를 유지하고 REPORT_TO_READ에 저장 JSON 경로를 넣습니다.
정확도/MRR와 NLL을 함께 비교하세요. gap이나 입력 최대값이 작아지는 것만으로 성공은 아닙니다.


In [ ]:
REPORT_TO_READ = None
if REPORT_TO_READ:
    path = Path(REPORT_TO_READ)
    if not path.is_absolute():
        path = ROOT / path
    loaded = json.loads(path.read_text(encoding="utf-8"), parse_constant=shared["reject_constant"])
    require(loaded["reportSchemaVersion"] == 1 and loaded["experimentId"] == EXPERIMENT_ID
            and loaded["runMode"] == "full" and loaded["variants"] == list(VARIANTS)
            and loaded["sourceDefinitionSha256"] == definition_sha(SOURCE_NOTEBOOK)
            and loaded["previousDefinitionsSha256"] == PREVIOUS_DEFS_SHA
            and loaded["referenceSha256"] == REFERENCE_SHA, "STALE_OR_DIFFERENT_REPORT")
    full_report = loaded


In [ ]:
if full_report is not None:
    show_report(full_report)


## 5. 한계 / 다음 gate
- 기존 동일 Validation을 반복 사용하는 개발 비교입니다. 독립 최종 Test/개인 복제/실제 게임 성능을 주장하지 않습니다.
- alpha는 같지만 feature scale에 따라 raw 입력 기준 L2 제약도 달라집니다. 스케일링 변경의 효용을 확인하며 특정 픽셀의 암기를 단독 원인으로 확정하지 않습니다.
- 후보가 좋아져도 희소2133-class/OOV·관측 시점·합법성/도달 가능성·현재40행 호환성 문제를 해결한 것은 아닙니다.
- 같은 seed로 각 조건1회이며 새 조건의 반복 결정성/수렴/동결 검증은 아닙니다. 조기 종료/온도 보정/지형 feature/출력 구조를 함께 바꾸지 않습니다.
- 출력은 report뿐이고 가중치·runtime·추가 수집을 만들지 않습니다. 다음은 사용자 Full 결과 해석이며 후속 실험/commit을 자동 시작하지 않습니다.
